In [1]:
# importing libraries
import pandas as pd
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
    set_seed
)
import numpy as np
from sklearn.metrics import classification_report
from datasets import load_from_disk
import evaluate

In [2]:
# Loading dataset
dataset_dict = load_from_disk("E:/PROJECTS/Privacy-Risk-Analyser/data/privacy_ner_dataset")

In [3]:
# Loading tokenizer and define label list
model_checkpoint = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

label_list = dataset_dict["train"].features["labels"].feature.names
label_to_id = {l: i for i, l in enumerate(label_list)}
num_labels = len(label_list)

In [4]:
# Tokenize with aligned labels
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


# Mapping dataset
tokenized_datasets = dataset_dict.map(tokenize_and_align_labels, batched=True)

In [5]:
# Loading model
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=num_labels)

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:

data_collator = DataCollatorForTokenClassification(tokenizer)

# Metrics
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_labels = [
        [label_list[l] for l in label if l != -100]
        for label in labels
    ]
    true_preds = [
        [label_list[p] for p, l in zip(pred_row, label_row) if l != -100]
        for pred_row, label_row in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }



In [7]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    do_eval=True,
    do_train=True,
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10
)


# Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Train
trainer.train()


C:\Users\Dell\AppData\Local\Temp\ipykernel_27868\3154479411.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
10,1.705200
20,0.274300
30,0.194600
40,0.134900
50,0.124000
60,0.092500
70,0.095700
80,0.082600
90,0.067700
100,0.072500


TrainOutput(global_step=168, training_loss=0.19133368134498596, metrics={'train_runtime': 4324.2397, 'train_samples_per_second': 0.308, 'train_steps_per_second': 0.039, 'total_flos': 344431335686256.0, 'train_loss': 0.19133368134498596, 'epoch': 3.0})

In [8]:
trainer.save_model("E:/PROJECTS/Privacy-Risk-Analyser/models/xlm-roberta-ner")
tokenizer.save_pretrained("E:/PROJECTS/Privacy-Risk-Analyser/models/xlm-roberta-ner")

('E:/PROJECTS/Privacy-Risk-Analyser/models/xlm-roberta-ner\\tokenizer_config.json',
 'E:/PROJECTS/Privacy-Risk-Analyser/models/xlm-roberta-ner\\special_tokens_map.json',
 'E:/PROJECTS/Privacy-Risk-Analyser/models/xlm-roberta-ner\\tokenizer.json')